# Limpeza dos dados — Grocery Sales Dataset
Segundo notebook do projeto. Ele:
1. **Puxa** os dados brutos da pasta `dados/`.
2. **Limpa** cada tabela com base nos problemas encontrados na EDA (`notebook.ipynb`).
3. **Salva** as tabelas tratadas na pasta `dados_limpos/`.

### Problemas identificados na EDA e tratamento aplicado
| Problema | Onde | Tratamento |
|---|---|---|
| `TotalPrice` 100% zerada | `sales` | Recalcular `Quantity × Price × (1 - Discount)` |
| `SalesDate` ausente (~1%, 67.526) | `sales` | Remover linhas sem data |
| `CountryCode` nulo (1) | `countries` | Preencher com `"??"` |
| `MiddleInitial` nulo (977) | `customers` | Preencher com `""` |
| Chaves órfãs (produto/cliente inexistente) | `sales` | Remover linhas sem correspondência |
| Descontos inválidos (fora de 0/0,1/0,2) | `sales` | Validar e remover |
| Duplicatas | todas | Remover |

## 1. Puxar os dados brutos

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ORIGEM = Path("dados")            # dados brutos
DESTINO = Path("dados_limpos")    # saida limpa
DESTINO.mkdir(exist_ok=True)

categories = pd.read_csv(ORIGEM/"categories.csv")
cities     = pd.read_csv(ORIGEM/"cities.csv")
countries  = pd.read_csv(ORIGEM/"countries.csv")
customers  = pd.read_csv(ORIGEM/"customers.csv")
employees  = pd.read_csv(ORIGEM/"employees.csv", parse_dates=["BirthDate", "HireDate"])
products   = pd.read_csv(ORIGEM/"products.csv", parse_dates=["ModifyDate"])

dtypes_sales = {"SalesID":"int64", "SalesPersonID":"int32", "CustomerID":"int32",
                "ProductID":"int32", "Quantity":"int32", "Discount":"float32",
                "TotalPrice":"float32"}
sales = pd.read_csv(ORIGEM/"sales.csv", dtype=dtypes_sales, parse_dates=["SalesDate"])

print("Dados brutos carregados:")
for nome, df in [("categories",categories),("cities",cities),("countries",countries),
                 ("customers",customers),("employees",employees),("products",products),("sales",sales)]:
    print(f"  {nome:11s} {len(df):>10,} linhas")

Dados brutos carregados:
  categories          11 linhas
  cities              96 linhas
  countries          206 linhas
  customers       98,759 linhas
  employees           23 linhas
  products           452 linhas
  sales        6,758,125 linhas


## 2. Limpeza das tabelas de dimensão

In [2]:
# --- countries: CountryCode nulo ---
n = countries.CountryCode.isna().sum()
countries["CountryCode"] = countries["CountryCode"].fillna("??")
print(f"countries: {n} CountryCode nulo(s) preenchido(s) com '??'")

# --- customers: MiddleInitial nulo ---
n = customers.MiddleInitial.isna().sum()
customers["MiddleInitial"] = customers["MiddleInitial"].fillna("")
print(f"customers: {n} MiddleInitial nulo(s) preenchido(s) com ''")

# --- remover duplicatas em todas as dimensoes ---
for nome, df in {"categories":categories,"cities":cities,"countries":countries,
                 "customers":customers,"employees":employees,"products":products}.items():
    antes = len(df)
    df.drop_duplicates(inplace=True)
    if antes != len(df):
        print(f"{nome}: {antes-len(df)} duplicata(s) removida(s)")

# --- products: garantir preco valido (>0) ---
prod_invalidos = (products.Price <= 0).sum()
print(f"products: {prod_invalidos} produto(s) com preco <= 0")
products = products[products.Price > 0].copy()

countries: 1 CountryCode nulo(s) preenchido(s) com '??'
customers: 977 MiddleInitial nulo(s) preenchido(s) com ''
products: 0 produto(s) com preco <= 0


## 3. Limpeza da tabela `sales`

In [3]:
linhas0 = len(sales)
log = []

# 3.1 remover duplicatas exatas e por SalesID
sales = sales.drop_duplicates()
sales = sales.drop_duplicates(subset="SalesID")
log.append(("duplicatas", linhas0 - len(sales)))

# 3.2 remover linhas sem data
antes = len(sales)
sales = sales.dropna(subset=["SalesDate"])
log.append(("SalesDate ausente", antes - len(sales)))

# 3.3 validar Quantity (>0) e Discount (valores esperados)
antes = len(sales)
sales = sales[sales.Quantity > 0]
log.append(("Quantity <= 0", antes - len(sales)))

# Discount vem em float32 (ex.: 0.10000000149) -> normalizar e validar com tolerancia
antes = len(sales)
descontos_validos = [0.0, 0.1, 0.2]
sales["Discount"] = sales["Discount"].astype("float64").round(2)
valido = np.isclose(sales["Discount"].values[:, None], descontos_validos, atol=1e-6).any(axis=1)
sales = sales[valido]
log.append(("Discount invalido", antes - len(sales)))

# 3.4 remover chaves orfas (produto/cliente/vendedor inexistentes)
antes = len(sales)
sales = sales[sales.ProductID.isin(products.ProductID)]
log.append(("ProductID orfao", antes - len(sales)))

antes = len(sales)
sales = sales[sales.CustomerID.isin(customers.CustomerID)]
log.append(("CustomerID orfao", antes - len(sales)))

antes = len(sales)
sales = sales[sales.SalesPersonID.isin(employees.EmployeeID)]
log.append(("SalesPersonID orfao", antes - len(sales)))

display(pd.DataFrame(log, columns=["motivo", "linhas_removidas"]))
print(f"sales: {linhas0:,} -> {len(sales):,} linhas ({(1-len(sales)/linhas0)*100:.2f}% removido)")

,motivo,linhas_removidas
0,duplicatas,0
1,SalesDate ausente,67526
2,Quantity <= 0,0
3,Discount invalido,0
4,ProductID orfao,0
5,CustomerID orfao,0
6,SalesPersonID orfao,0


sales: 6,758,125 -> 6,690,599 linhas (1.00% removido)


## 4. Recalcular `TotalPrice` (coluna estava 100% zerada)

In [4]:
# trazer o preco do produto e recomputar o total
sales = sales.merge(products[["ProductID", "Price", "CategoryID"]], on="ProductID", how="left")

sales["UnitPrice"] = sales["Price"].astype("float32")
sales["TotalPrice"] = (sales["Quantity"] * sales["Price"] * (1 - sales["Discount"])).astype("float32")
sales.drop(columns=["Price"], inplace=True)

print("TotalPrice recalculado.")
print(f"  Receita total: R$ {sales['TotalPrice'].sum():,.2f}")
print(f"  Ticket medio:  R$ {sales['TotalPrice'].mean():,.2f}")
print(f"  Zeros restantes em TotalPrice: {(sales['TotalPrice']==0).sum()}")
display(sales.head())

TotalPrice recalculado.
  Receita total: R$ 4,289,241,856.00
  Ticket medio:  R$ 641.08
  Zeros restantes em TotalPrice: 0


,SalesID,SalesPersonID,CustomerID,ProductID,Quantity,Discount,TotalPrice,SalesDate,TransactionNumber,CategoryID,UnitPrice
0,1,6,27039,381,7,0.0,309.635895,2018-02-05 07:38:25.430,FQL4S94E4ME1EZFTG42G,1,44.233700
1,2,16,25011,61,7,0.0,437.821991,2018-02-02 16:03:31.150,12UGLX40DJ1A5DTFBHB8,8,62.546001
2,3,13,94024,23,24,0.0,1896.441650,2018-05-03 19:31:56.880,5DT8RCPL87KI5EORO7B0,11,79.018402
3,4,8,73966,176,19,0.2,1236.013794,2018-04-07 14:43:55.420,R3DR9MLD5NR76VO17ULE,6,81.316704
4,5,10,32653,310,9,0.0,719.802002,2018-02-12 15:37:03.940,4BGS0Z5OMAZ8NDAFHHP3,9,79.977997


## 5. Verificação final de qualidade
Conferimos que não restaram nulos, duplicatas ou valores inválidos.

In [5]:
tabelas_limpas = {"categories":categories, "cities":cities, "countries":countries,
                  "customers":customers, "employees":employees, "products":products, "sales":sales}

print("Nulos restantes por tabela:")
for nome, df in tabelas_limpas.items():
    tot = int(df.isna().sum().sum())
    print(f"  {nome:11s} {tot} nulo(s)")

print("\nChecagens em sales:")
print("  TotalPrice negativo:", int((sales.TotalPrice < 0).sum()))
print("  SalesID duplicado:  ", int(sales.SalesID.duplicated().sum()))
print("  Datas fora de 2018: ", int((sales.SalesDate.dt.year != 2018).sum()))

Nulos restantes por tabela:
  categories  0 nulo(s)
  cities      0 nulo(s)
  countries   0 nulo(s)
  customers   0 nulo(s)
  employees   0 nulo(s)
  products    0 nulo(s)
  sales       0 nulo(s)

Checagens em sales:
  TotalPrice negativo: 0
  SalesID duplicado:   0
  Datas fora de 2018:  0


## 6. Salvar os dados limpos em `dados_limpos/`

In [6]:
for nome, df in tabelas_limpas.items():
    caminho = DESTINO / f"{nome}.csv"
    df.to_csv(caminho, index=False)
    print(f"salvo: {caminho}  ({len(df):,} linhas, {caminho.stat().st_size/1e6:.1f} MB)")

print("\nLimpeza concluida. Dados prontos em:", DESTINO.resolve())

salvo: dados_limpos\categories.csv  (11 linhas, 0.0 MB)
salvo: dados_limpos\cities.csv  (96 linhas, 0.0 MB)
salvo: dados_limpos\countries.csv  (206 linhas, 0.0 MB)
salvo: dados_limpos\customers.csv  (98,759 linhas, 4.5 MB)
salvo: dados_limpos\employees.csv  (23 linhas, 0.0 MB)
salvo: dados_limpos\products.csv  (452 linhas, 0.0 MB)
salvo: dados_limpos\sales.csv  (6,690,599 linhas, 612.7 MB)

Limpeza concluida. Dados prontos em: C:\workspace\codigos\PI-6-TIME1\dados_limpos
